# MNIST FPGA — training and INT8 quantization

This notebook trains the **784 → 16 → ReLU → 10** MNIST MLP and converts it into the integer representation used by the FPGA:

- pixels and weights: signed INT8
- biases and accumulators: signed INT32
- hidden activation: ReLU followed by a power-of-two right shift
- output: integer argmax, without softmax


## 1. Get the model branch

The cell clones the repository on the first run and performs a fast-forward pull on later runs.

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/mnist-fpga")

if repo_dir.exists():
    subprocess.run(
        ["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", "model"],
        check=True,
    )
else:
    subprocess.run(
        [
            "git", "clone", "--branch", "model", "--single-branch",
            "https://github.com/JKGiova/mnist-fpga.git", str(repo_dir),
        ],
        check=True,
    )

%cd /content/mnist-fpga

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Train the FP32 model — optional

The repository already contains training/artifacts/mnist-mlp-fp32.pt. Run this cell only when you want to retrain it.

In [ ]:
!python training/train.py \
    --epochs 25 \
    --batch-size 256 \
    --learning-rate 0.001

## 3. Quantize and test the model

quantize.py calibrates the hidden right shift on the MNIST training set, performs integer-only inference on all 10,000 test images, checks INT32 ranges, and exports the FPGA memory files.

In [ ]:
checkpoint = Path("training/artifacts/mnist-mlp-fp32.pt")
quantizer = Path("quantization/quantize.py")

assert checkpoint.exists(), f"Missing checkpoint: {checkpoint}"
assert quantizer.exists(), f"Missing quantizer: {quantizer}"

subprocess.run(
    ["python", str(quantizer), "--batch-size", "512"],
    check=True,
)

## 4. Inspect the quantization results

In [ ]:
import json

artifact_dir = Path("quantization/artifacts")
metadata_path = artifact_dir / "quantization.json"

with metadata_path.open(encoding="utf-8") as file:
    metadata = json.load(file)

metrics = metadata["metrics"]

print(f"FP32 accuracy:         {metrics['fp32_accuracy']:.2f}%")
print(f"INT8 accuracy:         {metrics['int8_accuracy']:.2f}%")
print(f"Prediction agreement:  {metrics['prediction_agreement']:.2f}%")
print(f"Hidden right shift:    {metadata['hidden']['right_shift']}")
print(
    "FC1 accumulator:       "
    f"{metrics['fc1_accumulator_min']} to {metrics['fc1_accumulator_max']}"
)
print(
    "FC2 accumulator:       "
    f"{metrics['fc2_accumulator_min']} to {metrics['fc2_accumulator_max']}"
)

## 5. Validate the FPGA memory files

Each assertion checks the number of memory words and the hexadecimal width expected by the Verilog memories.

In [ ]:
memory_layout = {
    "fc1-weights.mem": (784, 32),  # 128 bits
    "fc1-bias.mem": (16, 8),      # 32 bits
    "fc2-weights.mem": (16, 20),   # 80 bits
    "fc2-bias.mem": (10, 8),      # 32 bits
}

for filename, (expected_lines, expected_hex_width) in memory_layout.items():
    path = artifact_dir / filename
    lines = path.read_text(encoding="ascii").splitlines()

    assert len(lines) == expected_lines, (
        f"{filename}: expected {expected_lines} lines, got {len(lines)}"
    )
    assert all(len(line) == expected_hex_width for line in lines), (
        f"{filename}: invalid hexadecimal word width"
    )
    assert all(
        all(character in "0123456789abcdefABCDEF" for character in line)
        for line in lines
    ), f"{filename}: contains a non-hexadecimal character"

    print(f"PASS  {filename}: {len(lines)} words")

## 6. Download the INT8 checkpoint and FPGA exports

In [ ]:
import shutil
from google.colab import files

archive_path = shutil.make_archive(
    "/content/mnist-fpga-int8-artifacts",
    "zip",
    root_dir=artifact_dir,
)

print("Created:", archive_path)
files.download(archive_path)